In [2]:
import orekit
import os
import zipfile

orekit.initVM()

from orekit.pyhelpers import download_orekit_data_curdir, setup_orekit_curdir

data_path = os.environ["LUPNT_DATA_PATH"]
base_path = os.path.dirname(data_path)
zip_path = os.path.join(base_path, "orekit-data.zip")
folder_path = os.path.join(base_path, "orekit-data-master")
if not os.path.exists(folder_path):
    if not os.path.exists(zip_path):
        download_orekit_data_curdir(zip_path)
    # Expand zip
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(base_path)

setup_orekit_curdir(filename=folder_path)

OpenJDK 64-Bit Server VM warning: Attempt to protect stack guard pages failed.
OpenJDK 64-Bit Server VM warning: Attempt to deallocate stack guard pages failed.


In [3]:
from math import radians, degrees

from org.orekit.orbits import KeplerianOrbit, PositionAngleType
from org.orekit.propagation.analytical import KeplerianPropagator
from org.orekit.time import AbsoluteDate, TimeScalesFactory
from org.orekit.utils import Constants
from org.orekit.frames import FramesFactory
from org.orekit.orbits import OrbitType

from org.orekit.propagation.numerical import NumericalPropagator
from org.hipparchus.ode.nonstiff import ClassicalRungeKuttaIntegrator
from org.orekit.propagation import SpacecraftState
from org.orekit.forces.gravity.potential import GravityFieldFactory, SHAFormatReader
from org.orekit.forces.gravity import HolmesFeatherstoneAttractionModel, ThirdBodyAttraction
from org.orekit.bodies import CelestialBodyFactory

In [4]:
a = 6541.4e3  # [m] Semi-major axis
e = 0.6000  # [--] Eccentricity
i = radians(56.2)  # [deg] Inclination
O = radians(0.0)  # [deg] Right ascension of the ascending node
w = radians(90.0)  # [deg] Argument of perigee
M = radians(0.0)  # [deg] Mean anomaly

UTC = TimeScalesFactory.getUTC()
t0_utc = AbsoluteDate(2020, 1, 1, 12, 0, 0.000, UTC)

MOON = CelestialBodyFactory.getMoon()
SUN = CelestialBodyFactory.getSun()
EARTH = CelestialBodyFactory.getEarth()
MOON_CI = MOON.getInertiallyOrientedFrame()

coe_mi = KeplerianOrbit(a, e, i, w, O, M, PositionAngleType.MEAN, MOON_CI, t0_utc, Constants.WGS84_EARTH_MU)
coe_mi

<KeplerianOrbit: Keplerian parameters: {a: 6541400.0; e: 0.6; i: 56.2; pa: 90.0; raan: 0.0; v: 0.0;}>

In [5]:
Dt = 10.0
t_final = 7 * 24 * 3600.0
N_steps = int(t_final / Dt)

integrator = ClassicalRungeKuttaIntegrator(Dt)

In [6]:
satellite_mass = 100.0  # The models need a spacecraft mass, unit kg.
state0 = SpacecraftState(coe_mi, satellite_mass)

prop = NumericalPropagator(integrator)
prop.setOrbitType(OrbitType.CARTESIAN)
prop.setInitialState(state0)

GravityFieldFactory.clearPotentialCoefficientsReaders()
GravityFieldFactory.addPotentialCoefficientsReader(SHAFormatReader("sha.grgm1200a_50x50", True))
grav = GravityFieldFactory.getNormalizedProvider(5, 5)
prop.addForceModel(HolmesFeatherstoneAttractionModel(MOON.getBodyOrientedFrame(), grav))
prop.addForceModel(ThirdBodyAttraction(SUN))
prop.addForceModel(ThirdBodyAttraction(EARTH))

In [8]:
state = prop.propagate(t0_utc, t0_utc.shiftedBy(Dt))
state, state.getFrame()

(<SpacecraftState: SpacecraftState{orbit=Cartesian parameters: {P(-156063.9249536245, 1453962.3036130415, 2171902.9586773813), V(-15594.830762374682, -323.5426584854969, -483.3022561334056)}, attitude=org.orekit.attitudes.Attitude@205bed61, mass=100.0, additional={}, additionalDot={}}>,
 <Frame: Moon/inertial>)

In [9]:
import numpy as np
import time

rvs = np.zeros((N_steps, 6))
t_start = time.time()
for i in range(N_steps):
    state = prop.propagate(t0_utc.shiftedBy(i * Dt))
    rvs[i, :3] = np.array(state.getPVCoordinates().getPosition().toArray())
    rvs[i, 3:] = np.array(state.getPVCoordinates().getVelocity().toArray())
t_elapsed = time.time() - t_start
rvs /= 1e3
print(f"Elapsed time: {t_elapsed:.3f} s")
print(f"N_steps: {N_steps}")

Elapsed time: 2.959 s
N_steps: 60480


In [10]:
import plotly.graph_objects as go
import plotly.io as pio
pio.templates.default = "plotly_white"

n = 10
fig = go.Figure()
fig.add_trace(go.Scatter3d(x=rvs[::n, 0], y=rvs[::n, 1], z=rvs[::n, 2], mode='lines', name='Orbit'))
fig.update_layout(scene=dict(aspectmode='data'))
fig.show()
